# Puente Rectificador Monofásico Completamente Controlado  

Representación del comportamiento de un rectificador monofásico de onda completa con tiristores, mostrando las variaciones de corriente y voltaje en sus principales componentes.  

## Elementos Clave  

### 🔹 Corrientes en el sistema  
- **\(i_L\)** → Corriente en la carga  
- **\(i_{T1}, i_{T2}, i_{T3}, i_{T4}\)** → Corriente en los tiristores  
- **\(i_s\)** → Corriente en el transformador  

**Muestra el efecto del ángulo de disparo (\(\alpha\)) en la forma de onda de la corriente.**  

### 🔹 Voltajes en los componentes  
- **\(V_o\)** → Voltaje de salida  
- **\(V_{T1}, V_{T2}, V_{T3}, V_{T4}\)** → Voltaje en los tiristores  
- **\(V_L\)** → Voltaje en la carga  

**Permite analizar la influencia de \(\alpha\) en la entrega de potencia.**  

## ⚙️ Control de Fase  
El ajuste del ángulo de disparo \(\alpha\) regula la potencia suministrada a la carga, siendo clave en sistemas de control de voltaje.  

**Ideal para entender el funcionamiento y aplicaciones del rectificador monofásico completamente controlado.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import bilinear, lfilter
import ipywidgets as widgets
from IPython.display import display

class MonofasicoSim:
    def __init__(self):
        # Variables iniciales
        self.alfa = 0.1
        self.semi = False
        self.R = 1
        self.L = 1  # En mH
        self.Vp = 1
        self.E = 1

        

        # Widgets
        self.alpha_slider = widgets.FloatSlider(value=self.alfa, min=0, max=1, step=0.01, description='Ángulo α')
        self.semi_toggle = widgets.Checkbox(value=self.semi, description='Semicontrolado')
        self.R_input = widgets.FloatText(value=self.R, description='R (Ω)')
        self.L_input = widgets.FloatText(value=self.L, description='L (mH)')
        self.Vp_input = widgets.FloatText(value=self.Vp, description='Vp (V)')
        self.E_input = widgets.FloatText(value=self.E, description='E (V)')
        
        # Observadores para actualizar la gráfica en tiempo real
        for widget in [self.alpha_slider, self.semi_toggle, self.R_input, self.L_input, self.Vp_input, self.E_input]:
            widget.observe(self.update_plot, names='value')
        
        # Contenedor de salida
        self.output = widgets.Output()

        # Mostrar widgets
        display(self.alpha_slider, self.semi_toggle, self.R_input, self.L_input, self.Vp_input, self.E_input, self.output)

        # Dibujar la gráfica inicial
        self.update_plot()

    def update_plot(self, _=None):
        """Actualiza la simulación y la gráfica."""
        self.alfa = self.alpha_slider.value * np.pi
        self.semi = self.semi_toggle.value
        self.R = self.R_input.value
        self.L = self.L_input.value / 1000  # Convertir a Henrios
        self.Vp = self.Vp_input.value
        self.E = self.E_input.value

        with self.output:
            self.output.clear_output(wait=True)
            
            t = np.linspace(0, 1, 10000)
            vi = self.Vp * np.sin(2 * np.pi * t)
            indice = int(self.alfa * (10000 / (2 * np.pi)))

            if self.semi:
                vo = np.zeros_like(vi)
                vo[indice:5000] = vi[indice:5000]
                vo[indice+5000:] = -vi[indice+5000:]
            else:
                vo = np.zeros_like(vi)
                vo[:indice] = -vi[:indice]
                vo[indice:5000+indice] = vi[indice:5000+indice]
                vo[indice+5000:] = -vi[indice+5000:]

            voo = np.tile(vo, 30)
            tt = np.linspace(0, 30/60, len(voo))
            
            if self.semi:
                voo[-1000:] = voo[-1000:] * np.linspace(1, 0, 1000)

            b, a = bilinear([1], [self.L, self.R], fs=60 * 10000)
            io = lfilter(b, a, voo - self.E)

            fig, ax = plt.subplots(figsize=(7, 4))
            seno_completo = self.Vp * np.sin(2 * np.pi * 60 * tt[280000:300000])
            ax.plot(tt[280000:300000], seno_completo, 'blue', linestyle='dotted', linewidth=1)
            ax.plot(tt[280000:300000], voo[280000:300000], 'g', label="v_o")

            ax2 = ax.twinx()
            ax2.plot(tt[280000:300000], io[280000:300000], 'r', label="i_o")
            ax2.set_ylabel("Corriente i_o [A]", color='r')
            ax2.tick_params(axis='y', colors='r')
            ax2.set_ylim(0, max(io[np.isfinite(io)]) + 0.5)

            ax.legend(loc="upper left")
            ax2.legend(loc="upper right")
            ax.set_xlabel("Tiempo [s]")
            ax.set_ylim(-self.Vp, self.Vp)
            ax.grid(True)
            
            plt.show()
    

# Ejecutar
MonofasicoSim()


FloatSlider(value=0.1, description='Ángulo α', max=1.0, step=0.01)

Checkbox(value=False, description='Semicontrolado')

FloatText(value=1.0, description='R (Ω)')

FloatText(value=1.0, description='L (mH)')

FloatText(value=1.0, description='Vp (V)')

FloatText(value=1.0, description='E (V)')

Output()